# Phase 1 bootstrap — `/ask` + `/conversation`

Exploratory notebook. Validates the real logic (models, auth, Firestore write) here first, against real credentials, before it lands in `app/gateway/`. Once this works end to end, the last cells use `%%writefile` to persist the real modules.

**Scope for this step** (`docs/build-order.md` Phase 1 items 1+2, bundled):
- `POST /ask` — real `AgentResponse` schema, real auth, **canned** `answer_markdown` (no LangGraph loop until Phase 3).
- `POST /conversation` — fully real: mints a UUID, writes a real `sessions/{conversation_id}` doc to Firestore.

**Not in scope here:** `GET /ask/status`, `POST /ask/respond`, `POST /ask/cancel` — the last two are Phase 3 work per the build order (they act on a loop that doesn't exist yet).

**This notebook is Layer 2-ish exploration, not a substitute for the checked-in test suite.** Per `docs/testing.md`, real Layer 1 (mocked I/O) pytest tests still get written once this code lands in `app/gateway/` — that's a separate step after this one, not skipped because it worked here.

In [15]:
import sys
from pathlib import Path

# Notebook's cwd is notebooks/, but app/ is a sibling of notebooks/ at the repo
# root — add it to sys.path so `from app...` imports resolve.
sys.path.insert(0, str(Path.cwd().parent))

import uuid
from datetime import datetime, timezone
from typing import Literal, TypedDict

import jwt
from jwt import PyJWKClient
from fastapi import FastAPI, Header, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel
from google.cloud import firestore, secretmanager

from app.config import GCP_PROJECT_ID, TENANT_ID, EXPECTED_AUDIENCE

print("GCP_PROJECT_ID:", GCP_PROJECT_ID)
print("TENANT_ID:", TENANT_ID)
print("EXPECTED_AUDIENCE:", EXPECTED_AUDIENCE)

GCP_PROJECT_ID: instacart-ml-model
TENANT_ID: 7e6d319c-ffb2-4bbf-8865-d2e7580a8998
EXPECTED_AUDIENCE: api://4b86032f-4507-4239-bf90-a9b1c33c571c


## Pydantic / TypedDict models

**Correction from the first draft:** `AgentResponse` and `Claim` are defined
in `.claude/rules/orchestrator.md` — "the schema of record" per
`gateway.md` — not `gateway.md` itself. `AskRequest`/`ConversationResponse`
are the ones actually specified in `gateway.md`, since those are genuinely
gateway-owned wire shapes.

**Dropped from this step: `PendingApproval` and `ToolCallRecord`.**
Verified: `PendingApproval` is **not** a field of `AgentResponse` — it's a
separate Firestore (`live_turns`) document, built conditionally inside the
`finalize` node only when `needs_approval` is true, and never sent to the
client. Nothing in this step's scope references either type — a canned
`/ask` response never sets `needs_approval=True` — and both are
orchestrator-owned by `orchestrator.md`'s own path scope (`app/orchestrator/**`).
They belong there, built for real alongside Phase 3's `finalize` node.

**Placement correction:** `AgentResponse`/`Claim` are written to
`app/orchestrator/models.py` below, not `app/gateway/models.py` — same
path-scoping reasoning. The gateway just imports `AgentResponse` for its
`response_model`.

In [16]:
# orchestrator.md — "implement exactly"
class Claim(BaseModel):
    text: str
    numeric_value: float | None = None
    source_tool_call_id: str | None = None


class AgentResponse(BaseModel):
    answer_markdown: str
    sources: list[str]
    needs_approval: bool = False
    chart_url: str | None = None
    claims: list[Claim] = []
    suggested_follow_ups: list[str] = []
    iteration_cap_hit: bool = False
    pending_query: str | None = None
    estimated_cost: str | None = None
    cost_cap_exceeded: bool = False


# gateway.md
class AskRequest(BaseModel):
    question: str
    image_base64: str | None = None
    filter_context: list[dict] = []
    active_page: str | None = None
    conversation_id: str


class ConversationResponse(BaseModel):
    conversation_id: str


# Quick sanity check — construct one of each, make sure nothing's missing
print(ConversationResponse(conversation_id=str(uuid.uuid4())).model_dump_json())
print(AgentResponse(answer_markdown="test", sources=[]).model_dump_json())

{"conversation_id":"3595d045-0f56-4b88-a026-25843f3452d7"}
{"answer_markdown":"test","sources":[],"needs_approval":false,"chart_url":null,"claims":[],"suggested_follow_ups":[],"iteration_cap_hit":false,"pending_query":null,"estimated_cost":null,"cost_cap_exceeded":false}


## `get_secret()` — new code needed for auth, doesn't exist in `app/config.py` yet

Per `local-dev-environment-setup.md` Step 15 item 4. **Decision, flagged rather than made silently:** this ends up living in `app/config.py` (matches where the doc places it and where the other config values already are), but because `config.py` already has real content, the persist step below appends it with `Edit`, not `%%writefile` — `%%writefile` overwrites the whole file, which would clobber `TENANT_ID`/`GCP_PROJECT_ID`/everything else already there.

In [17]:
def get_secret(secret_id: str, project_id: str, version: str = "latest") -> str:
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version}"
    response = client.access_secret_version(request={"name": name})
    return response.payload.data.decode("UTF-8")


# Real call — proves the SA/user has secretAccessor, same check as Step 15 item 3
gateway_api_key = get_secret("gateway-api-key", GCP_PROJECT_ID)
print("Fetched gateway-api-key, length:", len(gateway_api_key))

Fetched gateway-api-key, length: 43


## Auth validation

Straight from `.claude/rules/gateway.md`. Two factors: the API key proves *knows a shared secret*, the JWT proves *is this person, right now*.

In [18]:
_jwks_client = PyJWKClient(
    f"https://login.microsoftonline.com/{TENANT_ID}/discovery/v2.0/keys"
)


def validate_entra_token(authorization: str) -> dict:
    if not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing bearer token")
    token = authorization.removeprefix("Bearer ")
    try:
        signing_key = _jwks_client.get_signing_key_from_jwt(token)
        return jwt.decode(
            token, signing_key.key, algorithms=["RS256"],
            audience=EXPECTED_AUDIENCE,
            issuer=f"https://login.microsoftonline.com/{TENANT_ID}/v2.0",
        )
    except jwt.PyJWTError as e:
        raise HTTPException(401, f"Invalid token: {e}")


def validate_api_key(x_api_key: str) -> None:
    if x_api_key != gateway_api_key:
        raise HTTPException(401, "Invalid API key")

## Testing `validate_entra_token` without a real Power Apps sign-in

There's no real user token yet — the connector/Power Apps app that would produce one doesn't exist until later in Phase 1 (chicken-and-egg, same situation Step 14 B4 already named). So this signs a **local test JWT** with a throwaway RSA key and monkeypatches the JWKS client to serve its public key — this is genuinely how the real Layer 1 pytest test will work too (`docs/testing.md`: no real Entra dependency, self-contained).

**Alternative, if you want a real Entra-issued token instead:** MSAL's device-code flow against the connector app registration (Step 14 B) would mint a genuine delegated token right now, no Power Apps needed. Didn't build that here since it adds a live interactive-login step to re-run every time this notebook resets; say the word if you'd rather test against a real token.

In [19]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization
import time

_test_private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
_test_kid = "test-key-1"


def _make_test_token(*, audience=EXPECTED_AUDIENCE, issuer=None, expired=False) -> str:
    now = int(time.time())
    claims = {
        "oid": "11111111-2222-3333-4444-555555555555",
        "aud": audience,
        "iss": issuer or f"https://login.microsoftonline.com/{TENANT_ID}/v2.0",
        "iat": now,
        "exp": now - 10 if expired else now + 3600,
    }
    return jwt.encode(claims, _test_private_key, algorithm="RS256", headers={"kid": _test_kid})


class _FakeSigningKey:
    def __init__(self, key):
        self.key = key


# Monkeypatch: serve the test key's public half instead of hitting Entra's real JWKS endpoint
_jwks_client.get_signing_key_from_jwt = lambda token: _FakeSigningKey(
    _test_private_key.public_key()
)

# Happy path
claims = validate_entra_token(f"Bearer {_make_test_token()}")
print("OK, claims:", claims)

# Failure paths — confirm they actually reject
for label, token_fn in [
    ("expired", lambda: _make_test_token(expired=True)),
    ("wrong audience", lambda: _make_test_token(audience="api://wrong-app")),
]:
    try:
        validate_entra_token(f"Bearer {token_fn()}")
        print(f"FAIL — {label} token was accepted, should have been rejected")
    except HTTPException as e:
        print(f"OK — {label} correctly rejected: {e.detail}")

OK, claims: {'oid': '11111111-2222-3333-4444-555555555555', 'aud': 'api://4b86032f-4507-4239-bf90-a9b1c33c571c', 'iss': 'https://login.microsoftonline.com/7e6d319c-ffb2-4bbf-8865-d2e7580a8998/v2.0', 'iat': 1789520126, 'exp': 1789523726}
OK — expired correctly rejected: Invalid token: Signature has expired
OK — wrong audience correctly rejected: Invalid token: Audience doesn't match


## `POST /conversation` — real logic, real Firestore write

Per `.claude/rules/gateway.md`: mints a UUID, creates `sessions/{conversation_id}` with `user_id` from the token's `oid`, empty `recent_messages`, `last_activity_at`. No LLM, no tools — nothing here should be canned.

**`firestore.AsyncClient`, not `firestore.Client`** — an `async def` route handler blocks the whole event loop if the I/O inside it isn't genuinely non-blocking; `gateway.md`'s own `assert_owns_conversation` example already uses `await db...get()`, so this matches the documented pattern, not a new one.

In [20]:
_db = firestore.AsyncClient(project=GCP_PROJECT_ID)


async def create_conversation(user_id: str) -> str:
    conversation_id = str(uuid.uuid4())
    await _db.collection("sessions").document(conversation_id).set({
        "user_id": user_id,
        "recent_messages": [],
        "last_activity_at": datetime.now(timezone.utc),
    })
    return conversation_id


# Real write against real Firestore — then read it back to prove it landed
test_conv_id = await create_conversation(claims["oid"])
doc = await _db.collection("sessions").document(test_conv_id).get()
print("Created:", test_conv_id)
print("Read back:", doc.to_dict())

# Clean up — this is throwaway test data, not a real conversation
await _db.collection("sessions").document(test_conv_id).delete()
print("Cleaned up test doc.")

Created: 98b8177d-5580-4554-82c8-6373f756f6b2
Read back: {'last_activity_at': DatetimeWithNanoseconds(2026, 9, 16, 0, 55, 27, 31555, tzinfo=datetime.timezone.utc), 'recent_messages': [], 'user_id': '11111111-2222-3333-4444-555555555555'}
Cleaned up test doc.


## Ownership check — every endpoint taking a `conversation_id`, except `/conversation` itself

Per `gateway.md`: a `conversation_id` is just a string — nothing stops a caller sending someone else's. Verifies the token's `oid` matches the stored `user_id`. **404, not 403** — a wrong ID must be indistinguishable from a made-up one; a 403 would leak that the ID exists. Was missing from `/ask` (a real gap, not deferred scope) — added here now.

In [22]:
async def assert_owns_conversation(conversation_id: str, claims: dict) -> None:
    snap = await _db.collection("sessions").document(conversation_id).get()
    if not snap.exists or snap.to_dict().get("user_id") != claims["oid"]:
        raise HTTPException(404, "Conversation not found.")


# Real Firestore writes for both cases, then the actual checks
mine_id = await create_conversation(claims["oid"])
theirs_id = await create_conversation("someone-elses-oid")

await assert_owns_conversation(mine_id, claims)
print("OK — owner matches, no exception")

try:
    await assert_owns_conversation(theirs_id, claims)
    print("FAIL — should have rejected someone else's conversation")
except HTTPException as e:
    print(f"OK — wrong owner correctly rejected: {e.status_code} {e.detail}")

try:
    await assert_owns_conversation("00000000-0000-0000-0000-000000000000", claims)
    print("FAIL — should have rejected a nonexistent conversation")
except HTTPException as e:
    print(f"OK — nonexistent conversation correctly rejected: {e.status_code} {e.detail}")

# Clean up
await _db.collection("sessions").document(mine_id).delete()
await _db.collection("sessions").document(theirs_id).delete()

OK — owner matches, no exception
OK — wrong owner correctly rejected: 404 Conversation not found.
OK — nonexistent conversation correctly rejected: 404 Conversation not found.


DatetimeWithNanoseconds(2026, 9, 16, 0, 56, 28, 108022, tzinfo=datetime.timezone.utc)

## `POST /ask` — real schema, canned content

Real `AgentResponse`, real auth already wired above — but `answer_markdown` is fixed. No LangGraph loop exists until Phase 3, so there's nothing real to compute yet.

In [23]:
def ask(request: AskRequest, claims: dict) -> AgentResponse:
    return AgentResponse(
        answer_markdown=f"Echo: {request.question}",
        sources=[],  # real dispatch-order labels land with the loop in Phase 3
    )


result = ask(AskRequest(question="test question", conversation_id=test_conv_id), claims)
print(result.model_dump_json(indent=2))

{
  "answer_markdown": "Echo: test question",
  "sources": [],
  "needs_approval": false,
  "chart_url": null,
  "claims": [],
  "suggested_follow_ups": [],
  "iteration_cap_hit": false,
  "pending_query": null,
  "estimated_cost": null,
  "cost_cap_exceeded": false
}


## Wiring both into a real FastAPI app, tested via `TestClient`

`TestClient` calls the ASGI app in-process — no need to run `uvicorn` separately to test this interactively.

In [ ]:
# Fresh client for this section — see below for why `with TestClient(...)`
# matters, not just `TestClient(...)`.
_db = firestore.AsyncClient(project=GCP_PROJECT_ID)

app = FastAPI()


@app.post("/conversation", response_model=ConversationResponse)
async def post_conversation(authorization: str = Header(...), x_api_key: str = Header(...)):
    validate_api_key(x_api_key)
    token_claims = validate_entra_token(authorization)
    return ConversationResponse(conversation_id=await create_conversation(token_claims["oid"]))


@app.post("/ask", response_model=AgentResponse)
async def post_ask(
    body: AskRequest, authorization: str = Header(...), x_api_key: str = Header(...)
):
    validate_api_key(x_api_key)
    token_claims = validate_entra_token(authorization)
    await assert_owns_conversation(body.conversation_id, token_claims)
    return ask(body, token_claims)


headers = {"Authorization": f"Bearer {_make_test_token()}", "x-api-key": gateway_api_key}

# Set up "someone else's" conversation with the plain SYNC client — no
# await, no event loop involved at all, so it can't collide with whatever
# loop the portal below ends up using.
others_conv_id = str(uuid.uuid4())
firestore.Client(project=GCP_PROJECT_ID).collection("sessions").document(others_conv_id).set({
    "user_id": "someone-elses-oid", "recent_messages": [], "last_activity_at": datetime.now(timezone.utc),
})

# `with TestClient(app) as client:` — not just `client = TestClient(app)`.
# TestClient only keeps ONE persistent event loop (a "portal") alive across
# multiple requests when used as a context manager. Without `with`, it
# spins up a brand-new, throwaway loop for EACH individual `.post()` call
# and destroys it right after — so `_db`'s gRPC channel, bound on its first
# use, would be left pointing at an already-closed loop by the second call
# ("RuntimeError: Event loop is closed"). One `with` block = one shared
# loop = `_db` stays valid for every request inside it.
with TestClient(app) as client:
    conv_resp = client.post("/conversation", headers=headers)
    print("POST /conversation ->", conv_resp.status_code, conv_resp.json())
    new_conv_id = conv_resp.json()["conversation_id"]

    ask_resp = client.post(
        "/ask", headers=headers,
        json={"question": "How many orders last week?", "conversation_id": new_conv_id},
    )
    print("POST /ask ->", ask_resp.status_code, ask_resp.json())

    bad_resp = client.post(
        "/ask", headers={**headers, "x-api-key": "wrong"},
        json={"question": "x", "conversation_id": new_conv_id},
    )
    print("POST /ask with bad key ->", bad_resp.status_code, "(expect 401)")

    ownership_resp = client.post(
        "/ask", headers=headers, json={"question": "x", "conversation_id": others_conv_id},
    )
    print("POST /ask for someone else's conversation ->", ownership_resp.status_code, "(expect 404)")

# Clean up — plain sync client again, same reasoning: no event loop involved.
_sync_db = firestore.Client(project=GCP_PROJECT_ID)
_sync_db.collection("sessions").document(new_conv_id).delete()
_sync_db.collection("sessions").document(others_conv_id).delete()

## Persist to real files

Only run these once everything above checks out. Each `%%writefile` cell
**overwrites its target file completely** — fine for `app/orchestrator/models.py`
and `app/gateway/models.py`/`main.py` since they're new, but `app/config.py`
already has real content, so `get_secret()` needs to be appended by hand
instead (flagged above).

In [ ]:
%%writefile ../app/orchestrator/models.py
from pydantic import BaseModel


class Claim(BaseModel):
    text: str
    numeric_value: float | None = None
    source_tool_call_id: str | None = None


class AgentResponse(BaseModel):
    """The wire format the gateway returns. Schema of record:
    .claude/rules/orchestrator.md — implement exactly."""
    answer_markdown: str
    sources: list[str]
    needs_approval: bool = False
    chart_url: str | None = None
    claims: list[Claim] = []
    suggested_follow_ups: list[str] = []
    iteration_cap_hit: bool = False
    pending_query: str | None = None
    estimated_cost: str | None = None
    cost_cap_exceeded: bool = False

In [ ]:
%%writefile ../app/gateway/models.py
from pydantic import BaseModel


class AskRequest(BaseModel):
    question: str
    image_base64: str | None = None
    filter_context: list[dict] = []
    active_page: str | None = None
    conversation_id: str


class ConversationResponse(BaseModel):
    conversation_id: str

In [ ]:
%%writefile ../app/gateway/main.py
import uuid
from datetime import datetime, timezone
from functools import lru_cache

import jwt
from jwt import PyJWKClient
from fastapi import FastAPI, Header, HTTPException
from google.cloud import firestore

from app.config import GCP_PROJECT_ID, TENANT_ID, EXPECTED_AUDIENCE, get_secret
from app.gateway.models import AskRequest, ConversationResponse
from app.orchestrator.models import AgentResponse

app = FastAPI()
_jwks_client = PyJWKClient(
    f"https://login.microsoftonline.com/{TENANT_ID}/discovery/v2.0/keys"
)


# Lazy, not module-level globals: constructing these eagerly at import time
# means Layer 1 tests can't import this module without real credentials
# (google.cloud.firestore.AsyncClient() raises DefaultCredentialsError with
# none configured; get_secret() makes a real Secret Manager call). Deferring
# to first call — cached after that — keeps production behavior identical
# and makes both cleanly monkeypatchable in tests.
@lru_cache
def get_db() -> firestore.AsyncClient:
    return firestore.AsyncClient(project=GCP_PROJECT_ID)


@lru_cache
def get_gateway_api_key() -> str:
    return get_secret("gateway-api-key", GCP_PROJECT_ID)


def validate_entra_token(authorization: str) -> dict:
    if not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing bearer token")
    token = authorization.removeprefix("Bearer ")
    try:
        signing_key = _jwks_client.get_signing_key_from_jwt(token)
        return jwt.decode(
            token, signing_key.key, algorithms=["RS256"],
            audience=EXPECTED_AUDIENCE,
            issuer=f"https://login.microsoftonline.com/{TENANT_ID}/v2.0",
        )
    except jwt.PyJWTError as e:
        raise HTTPException(401, f"Invalid token: {e}")


def validate_api_key(x_api_key: str) -> None:
    if x_api_key != get_gateway_api_key():
        raise HTTPException(401, "Invalid API key")


async def create_conversation(user_id: str) -> str:
    conversation_id = str(uuid.uuid4())
    await get_db().collection("sessions").document(conversation_id).set({
        "user_id": user_id,
        "recent_messages": [],
        "last_activity_at": datetime.now(timezone.utc),
    })
    return conversation_id


async def assert_owns_conversation(conversation_id: str, claims: dict) -> None:
    snap = await get_db().collection("sessions").document(conversation_id).get()
    if not snap.exists or snap.to_dict().get("user_id") != claims["oid"]:
        raise HTTPException(404, "Conversation not found.")


@app.post("/conversation", response_model=ConversationResponse)
async def post_conversation(authorization: str = Header(...), x_api_key: str = Header(...)):
    validate_api_key(x_api_key)
    claims = validate_entra_token(authorization)
    return ConversationResponse(conversation_id=await create_conversation(claims["oid"]))


@app.post("/ask", response_model=AgentResponse)
async def post_ask(
    body: AskRequest, authorization: str = Header(...), x_api_key: str = Header(...)
):
    validate_api_key(x_api_key)
    claims = validate_entra_token(authorization)
    await assert_owns_conversation(body.conversation_id, claims)
    # Canned — real synthesis, sources, and claims land with the LangGraph
    # loop in Phase 3
    return AgentResponse(
        answer_markdown=f"Echo: {body.question}",
        sources=[],
    )

## Left to do after this notebook

1. Append `get_secret()` to `app/config.py` by hand (not `%%writefile`).
2. Write the real Layer 1 pytest tests for `app/gateway/main.py` (mocked
   Firestore client, self-signed-JWT auth tests — same pattern as the
   `auth-test` cell above, but as real fixtures in `tests/conftest.py` per
   `docs/testing.md`).
3. Confirm `uv run uvicorn app.gateway.main:app --reload` actually starts
   clean, not just `TestClient`.
4. `app/orchestrator/models.py` now holds `Claim`/`AgentResponse` ahead of
   the rest of the orchestrator — worth a one-line note in `docs/build-order.md`
   if Phase 3 planning assumed that module started empty.

http://127.0.0.1:8000/docs (Open docs)